# Multivariate adaptive shrinkage (MASH) mini-protocol
Prepare cross-condition summary statistics, fit a multivariate adaptive-shrinkage model, and estimate condition-specific effects and contrasts.

#### Miniprotocol Timing
This is the total duration for the selected route; module-specific timings appear on their respective pages.
Timing: TBD

## Overview
This mini-protocol organizes the MASH workflow into preprocessing, model fitting, and posterior analysis. Steps 1–2 call [`mash_preprocessing.ipynb`](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_preprocessing.html), step 3 calls [`mash_fit.ipynb`](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_fit.html), and steps 4–5 call [`mash_posterior.ipynb`](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_posterior.html). The covariance prior and residual variance used for fitting can be learned with [`mixture_prior.ipynb`](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mixture_prior.html).
Steps 1 and 2 are alternative preprocessing routes. Step 3 fits a model from prepared inputs, while steps 4–5 apply an existing model and summarize posterior contrasts.

## Steps
Choose a route before running commands; the commands are not one mandatory chain.
| Analysis goal | Commands to run, in order | Inputs |
|---|---|---|
| Build MASH-ready data from fine-mapping results and fit a model | 1 → 3 | `tests/fixtures/qtl_mini/fine_mapping_meta.tsv`; prepared covariance prior and residual-variance RDS files |
| Build random and null sets from tensorQTL results and fit a model | 2 → 3 | `input/finemapping/protocol_example.region`; `input/protocol_example.sumstats_list.txt`; prepared covariance prior and residual-variance RDS files |
| Apply an existing MASH model | 4 | `input/finemapping/protocol_example.analysis_units.txt`; `input/mash/protocol_example.mash_model.rds`; `input/twas/protocol_example.posterior_vhat.rds` |
| Plot posterior contrasts | 4 → 5 | Inputs for step 4 |

The fitting command uses `input/mash/protocol_example.EE.mash.rds`, `input/mash/protocol_example.EE.V_simple.rds`, and `input/mash/protocol_example.EE.prior.rds`. The prior and residual variance can be learned with the [mixture-prior module](https://statfungen.github.io/xqtl-protocol/mixture_prior.html); its covariance-component and variance-estimation workflows are alternatives and should be selected deliberately rather than run as a single chain.

### 1. [Prepare strong effects from fine-mapping results](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_preprocessing.html)

**What it does:** Converts fine-mapping posterior estimates into the strong-effect data used to learn or fit a MASH model.

**Timing**: TBD

In [ ]:
sos run pipeline/mash_preprocessing.ipynb susie_to_mash \
    --name protocol_example_mash \
    --fine_mapping_meta tests/fixtures/qtl_mini/fine_mapping_meta.tsv \
    --finemapping_column susie_path \
    --sig_p_cutoff 0.1 \
    --cwd output/mash_preprocessing

### 2. [Prepare random and null effects from tensorQTL results](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_preprocessing.html)

**What it does:** Samples random regions and extracts null effects from tensorQTL summary statistics to complement the strong-effect set. Use this route instead of step 1 when tensorQTL summary statistics are the starting point.

**Timing**: TBD

In [ ]:
sos run pipeline/mash_preprocessing.ipynb random_null_tensorqtl \
    --name protocol_example_mash \
    --region_file output/mash_preprocessing/protocol_example.region \
    --sum_files output/mash_preprocessing/protocol_example.sumstats_list.txt \
    --traits bulk_rnaseq \
    --cwd output/mash_preprocessing

### 3. [Fit the MASH model](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_fit.html)

**What it does:** Fits the multivariate adaptive-shrinkage model using prepared effects, a residual-variance estimate, and a covariance prior; `--compute-posterior` also stores posterior summaries for the fitted data.

**Timing**: TBD

In [ ]:
sos run pipeline/mash_fit.ipynb mash \
    --output-prefix protocol_example_mash \
    --data output/mash_fit/protocol_example.EE.mash.rds \
    --vhat-data output/mash_fit/protocol_example.EE.V_simple.rds \
    --prior-data output/mash_fit/protocol_example.EE.prior.rds \
    --effect-model EE \
    --compute-posterior \
    --cwd output/mash_fit

### 4. [Apply the fitted model](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_posterior.html)

**What it does:** Computes posterior effect estimates for each analysis unit using an existing MASH model. Adjust `--exclude-condition` only when specific conditions must be omitted.

**Timing**: ~30 sec (on toy dataset)

In [ ]:
sos run pipeline/mash_posterior.ipynb posterior \
    --cwd output/mash_posterior \
    --analysis-units output/mash_posterior/protocol_example.analysis_units.txt \
    --mash-model output/mash_posterior/protocol_example.mash_model.rds \
    --posterior-vhat-files output/mash_posterior/protocol_example.posterior_vhat.rds \
    --data-table-name strong \
    --exclude-condition 1 3

### 5. [Plot posterior contrasts](https://statfungen.github.io/xqtl-protocol/code/multivariate_genome/MASH/mash_posterior.html)

**What it does:** Summarizes and plots the condition contrasts produced by step 4.

**Timing**: TBD

In [ ]:
sos run pipeline/mash_posterior.ipynb posterior_contrast_plot \
    --cwd output/mash_posterior \
    --analysis-units output/mash_posterior/protocol_example.analysis_units.txt

## Output Files

| Step | Relative path | Contents |
|---:|---|---|
| 1 | `output/mash_preprocessing/protocol_example_mash.*.posterior_list` | Fine-mapping posterior lists prepared for MASH |
| 2 | `output/mash_preprocessing/protocol_example_mash.*` | Random/null effects and assembled preprocessing products |
| 3 | `output/mash_fit/protocol_example_mash.EE.mash_model.rds` | Fitted MASH model |
| 3 | `output/mash_fit/protocol_example_mash.EE.mash_model.posterior.rds` | Posterior summaries when `--compute-posterior` is enabled |
| 4 | `output/mash_posterior/*.posterior.rds` | Per-analysis-unit posterior estimates |
| 5 | `output/mash_posterior/*.posterior_sum.csv` | Combined posterior contrast summary |
| 5 | `output/mash_posterior/*.posterior_sum.png` | Posterior contrast plot |

Exact per-unit stems are derived from the analysis-units file. Review the module output sections before scripting downstream file discovery.

## Anticipated Results

The fitted model captures effect-sharing patterns across conditions while allowing condition-specific effects. Posterior outputs report shrunk effect estimates and uncertainty for each analysis unit; contrast summaries identify effects that differ between selected conditions.

Inspect the learned covariance components, mixture weights, model diagnostics, and posterior contrast distributions before downstream interpretation. Continue to feature-score workflows in the MASH posterior module only when the corresponding contrast, fine-mapping, or summary-statistic inputs are available.

## Command interface

In [ ]:
sos run pipeline/mash_preprocessing.ipynb -h

In [ ]:
sos run pipeline/mash_fit.ipynb -h

In [ ]:
sos run pipeline/mash_posterior.ipynb -h

In [ ]:
sos run pipeline/mixture_prior.ipynb -h